<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260224.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
%%writefile server.js

/*
  Express 웹 서버를 사용하여
  1) HTML 파일을 브라우저에 내려주고
  2) 폼 데이터를 받아
  3) 템플릿 HTML 파일을 읽어 값 치환 후
  4) 새로운 HTML 파일을 생성하여 응답하는 구조
*/

const express = require("express");   // 웹 서버 프레임워크
const path = require("path");         // 파일 경로 처리를 위한 Node 기본 모듈
const fs = require("fs");             // 파일 읽기/쓰기(File System) 모듈

const app = express();                // Express 애플리케이션 객체 생성

/*
  express.urlencoded 미들웨어
  ----------------------------------------------------
  HTML <form>이 기본적으로 보내는 데이터 형식은
  application/x-www-form-urlencoded 이다.

  이 미들웨어는 POST 요청으로 전달된 form 데이터를
  req.body 객체로 파싱해준다.

  예:
    customer=홍길동&item=커피&qty=2
  → req.body = { customer: "홍길동", item: "커피", qty: "2" }
*/
app.use(express.urlencoded({ extended: true }));

/*
  ----------------------------------------------------
  GET "/"
  ----------------------------------------------------
  브라우저가 http://localhost:3000 에 접속하면 실행된다.

  res.sendFile()은 지정한 HTML 파일을
  그대로 브라우저에 전달한다.

  즉, 서버가 HTML 파일을 렌더링하는 것이 아니라
  "파일을 전달"하면 브라우저가 렌더링한다.
*/
app.get("/", (req, res) => {
  res.sendFile(path.join(__dirname, "index.html"));
});

/*
  ----------------------------------------------------
  POST "/submit"
  ----------------------------------------------------
  사용자가 폼을 제출하면 실행된다.

  처리 흐름:
    1) 사용자가 입력한 값(req.body) 추출
    2) 주문서 템플릿 HTML 파일 읽기
    3) 템플릿의 {{변수}}를 실제 값으로 치환
    4) 완성된 HTML을 order.html 파일로 저장
    5) 생성된 HTML 파일을 브라우저에 응답
*/
app.post("/submit", (req, res) => {

  // 구조분해 할당으로 form 데이터 추출
  // req.body는 express.urlencoded 미들웨어가 만들어줌
  const { customer, item, qty } = req.body;

  /*
    order_template.html 파일을 읽는다.
    utf8로 읽어야 문자열(String)로 처리 가능하다.
  */
  const orderTemplateContent = fs.readFileSync(
    path.join(__dirname, "order_template.html"),
    "utf8"
  );

  /*
    템플릿 치환 작업
    ---------------------------------
    예:
      <p>주문자: {{customer}}</p>

    replaceAll을 이용해
    {{customer}} → 실제 입력값으로 교체
  */
  const completedOrderHtml = orderTemplateContent
    .replaceAll("{{customer}}", customer)
    .replaceAll("{{item}}", item)
    .replaceAll("{{qty}}", qty);

  /*
    완성된 HTML을 새로운 파일(order.html)로 저장한다.
    → 이 시점에 서버의 파일 시스템에 실제 파일이 생성됨
  */
  fs.writeFileSync(
    path.join(__dirname, "order.html"),
    completedOrderHtml
  );

  /*
    생성된 order.html 파일을 브라우저에 전송한다.
    응답은 문자열이 아니라 "HTML 파일"이다.
  */
  res.sendFile(path.join(__dirname, "order.html"));
});

/*
  ----------------------------------------------------
  서버 실행
  ----------------------------------------------------
  포트 3000에서 HTTP 서버를 시작한다.

  브라우저 접속 주소:
    http://localhost:3000
*/
app.listen(3000, () => {
  console.log("Server started at http://localhost:3000");
});

Overwriting server.js


In [26]:
%%writefile index.html

<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>주문 입력</title>
</head>
<body>
  <h1>주문 입력 화면</h1>

  <form method="POST" action="/submit">
    <p>
      주문자:
      <input name="customer" required>
    </p>

    <p>
      상품명:
      <input name="item" required>
    </p>

    <p>
      수량:
      <input type="number" name="qty" min="0" value="0" required>
    </p>

    <button type="submit">주문 접수</button>
  </form>
</body>
</html>

Overwriting index.html


In [27]:
%%writefile order_template.html

<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>주문서</title>
</head>
<body>
  <h1>주문서</h1>

  <p>주문자: {{customer}}</p>
  <p>상품명: {{item}}</p>
  <p>수량: {{qty}}</p>

  <hr>
  <a href="/">새 주문하기</a>
</body>
</html>

Overwriting order_template.html


In [28]:
!npm install express

⠙⠹⠸⠼⠴
up to date, audited 66 packages in 820ms
⠴
⠴22 packages are looking for funding
⠴  run `npm fund` for details
⠴
found 0 vulnerabilities
⠴

In [29]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 2s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [30]:
import subprocess
import time
from google.colab.output import eval_js

# 서버 백그라운드 실행
subprocess.Popen(["node", "server.js"])
time.sleep(2)

# Colab 내장 터널로 URL 출력
url = eval_js("google.colab.kernel.proxyPort(3000)")
print("접속 URL:", url)

접속 URL: https://3000-m-s-22vc49twyco8d-b.us-east1-2.prod.colab.dev


In [31]:
%%writefile index.html

<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>DOM Selection Minimal</title>
</head>
<body>
  <div id="title">Hello</div>
  <div class="box">A</div>
  <div class="box">B</div>

  <script>
    // ID 선택 (단일 요소)
    const title = document.getElementById("title");

    // 클래스 선택 (HTMLCollection)
    const boxes = document.getElementsByClassName("box");

    // CSS 선택자 (가장 권장)
    const firstBox = document.querySelector(".box");
    const allBoxes = document.querySelectorAll(".box");

    // 확인용 출력 (최소 구성)
    console.log("title:", title.textContent);
    console.log("boxes length (HTMLCollection):", boxes.length);
    console.log("firstBox:", firstBox.textContent);
    console.log("allBoxes length (NodeList):", allBoxes.length);
  </script>
</body>
</html>

Overwriting index.html


In [32]:
from IPython.display import HTML, display
display(HTML(open("/content/index.html", "r", encoding="utf-8").read()))

In [33]:
%%writefile index2.html

<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>DOM Selection Minimal</title>
</head>
<body>
 <ul id="list">
  <li>Item 1</li>
  <li>Item 2</li>
</ul>

  <script>
    const list = document.getElementById("list");

// 부모 노드
console.log(list.parentNode);

// 자식 노드들
console.log(list.children);

// 첫 번째 자식
console.log(list.firstElementChild);

// 다음 형제
console.log(list.firstElementChild.nextElementSibling);
  </script>
</body>
</html>

Overwriting index2.html


In [34]:
from IPython.display import HTML, display
display(HTML(open("/content/index2.html", "r", encoding="utf-8").read()))

In [35]:
%%writefile dom_example.html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Simple DOM Manipulation</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        #container { border: 1px solid #ccc; padding: 10px; min-height: 100px; margin-top: 10px; }
        .dom-element { background-color: #f0f0f0; margin: 5px; padding: 8px; border: 1px solid #ddd; }
    </style>
</head>
<body>
    <h2>간결한 DOM 조작 예제</h2>
    <button id="addBtn">요소 추가</button>
    <button id="removeBtn">요소 삭제</button>
    <div id="container">
        <!-- DOM 요소가 여기에 추가됩니다 -->
    </div>

    <script>
        document.addEventListener('DOMContentLoaded', () => {
            const addBtn = document.getElementById('addBtn');
            const removeBtn = document.getElementById('removeBtn');
            const container = document.getElementById('container');
            let elementCount = 0;

            // 요소 추가 기능
            addBtn.addEventListener('click', () => {
                elementCount++;
                const newElement = document.createElement('div');
                newElement.className = 'dom-element';
                newElement.textContent = `새로운 DOM 요소 ${elementCount}`;
                container.appendChild(newElement);
                console.log(`요소 추가됨: ${newElement.textContent}`);
            });

            // 요소 삭제 기능
            removeBtn.addEventListener('click', () => {
                if (container.lastChild) {
                    const removedElement = container.lastChild;
                    container.removeChild(removedElement);
                    console.log(`요소 삭제됨: ${removedElement.textContent}`);
                    elementCount--;
                } else {
                    console.log('삭제할 요소가 없습니다.');
                }
            });
        });
    </script>
</body>
</html>

Overwriting dom_example.html
